# Diffusion Models and Score-Based Generative Modeling
## From Gaussian noise to data — and back

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/diffusion_models.ipynb)

---

> **Prerequisites:** [`generative_models.ipynb`](generative_models.ipynb) (VAEs, ELBO, reparameterization), probability (Gaussian distributions, Markov chains), basic calculus. Familiarity with the forward pass and backpropagation is helpful but not required for this overview.

In [`generative_models.ipynb`](generative_models.ipynb) we saw two generative paradigms: VAEs (encode to a latent, decode probabilistically) and autoregressive models (factor $p(\mathbf{x}) = \prod_t p(x_t | x_{<t})$). Both have limitations — VAEs tend to produce blurry samples; autoregressive models are slow at generation time.

**Diffusion models** (Ho et al., 2020; Song & Ermon, 2019) are currently the dominant paradigm for high-quality generation (Stable Diffusion, DALL·E, Sora). Their key idea is deceptively simple: *learn to reverse a noise-adding process*. The mathematics is elegant, drawing on Markov chains, Gaussian statistics, and the theory of stochastic differential equations.

**Outline**
1. [The forward diffusion process](#part1)
2. [Score functions](#part2)
3. [The reverse process and DDPM](#part3)
4. [Training: denoising score matching](#part4)
5. [Extensions: from discrete to continuous time](#part5)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from ipywidgets import interact, IntSlider, FloatSlider
from scipy.special import logsumexp

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})
rng = np.random.default_rng(42)

# ── 2-D toy dataset: mixture of 3 Gaussians on the vertices of a triangle ──
MEANS = np.array([[0.0, 1.5], [-1.3, -0.75], [1.3, -0.75]])
SIGMA0 = 0.28          # within-cluster std

def sample_data(n, seed=0):
    rng_ = np.random.default_rng(seed)
    k = rng_.integers(0, 3, n)
    return MEANS[k] + rng_.standard_normal((n, 2)) * SIGMA0

X0 = sample_data(800)
print(f"Dataset: {X0.shape}  (2-D, 3 Gaussian clusters)")

<a id="part1"></a>
# Part 1: The Forward Diffusion Process

## 1.1 Gradually adding noise

The forward process is a fixed (non-learned) Markov chain that *gradually corrupts* data by adding small amounts of Gaussian noise at each of $T$ steps:

$$\boxed{q(\mathbf{x}_t \mid \mathbf{x}_{t-1}) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t\mathbf{I}\right), \qquad t = 1, \ldots, T.}$$

The scalars $0 < \beta_1 < \beta_2 < \cdots < \beta_T < 1$ form the **noise schedule**: small early, larger later. At step $t$, the mean is shrunk by $\sqrt{1-\beta_t}$ and independent Gaussian noise with variance $\beta_t$ is added.

## 1.2 Closed form: $q(\mathbf{x}_t \mid \mathbf{x}_0)$

A key algebraic fact: applying the Markov chain $t$ times gives a Gaussian with a closed form. Define $\alpha_t = 1 - \beta_t$ and $\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$. Then:

$$\boxed{q(\mathbf{x}_t \mid \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_t;\; \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0,\; (1-\bar{\alpha}_t)\mathbf{I}\right).}$$

Equivalently, via the reparameterization trick with $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$:

$$\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1 - \bar{\alpha}_t}\,\boldsymbol{\epsilon}.$$

As $t \to T$: $\bar{\alpha}_T \approx 0$, so $\mathbf{x}_T \approx \boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0},\mathbf{I})$. The data is completely destroyed. The reverse process must learn to undo this.

In [ ]:
# ── Noise schedule (linear, as in Ho et al. 2020) ───────────────────────────
T = 400
betas      = np.linspace(1e-4, 0.02, T)
alphas     = 1.0 - betas
alpha_bars = np.cumprod(alphas)           # ᾱ_t = ∏_{s=1}^t α_s

def forward_sample(x0, t):
    # x_t = sqrt(ᾱ_t) x_0 + sqrt(1-ᾱ_t) ε
    ab = alpha_bars[t]
    eps = rng.standard_normal(x0.shape)
    return np.sqrt(ab) * x0 + np.sqrt(1.0 - ab) * eps, eps

# ── Visualise the forward process at 5 time-steps ────────────────────────────
ts_show = [0, 40, 120, 250, 399]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))

for ax, t in zip(axes, ts_show):
    Xt, _ = forward_sample(X0, t)
    ax.scatter(Xt[:, 0], Xt[:, 1], c='#3498db', s=6, alpha=0.5, edgecolors='none')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ab = alpha_bars[t]
    ax.set_title(fr'$t={t}$''\n'fr'$\bar{{\alpha}}_t={ab:.3f}$', fontsize=11)

axes[0].set_title(r'$t=0$  (data)' + '\n' + r'$\bar{\alpha}_0=1.00$', fontsize=11)
axes[-1].set_title(r'$t=399$  (noise)' + '\n' + fr'$\bar{{\alpha}}_T={alpha_bars[-1]:.3f}$',
                   fontsize=11)
fig.suptitle('Forward diffusion: gradual corruption of the 3-cluster dataset', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def _plot_fwd_step(t=0):
    Xt, _ = forward_sample(X0, t)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
    axes[0].scatter(X0[:, 0], X0[:, 1], c='#3498db', s=8, alpha=0.5, edgecolors='none')
    axes[0].set_title(r'Original data  $\mathbf{x}_0$')
    axes[1].scatter(Xt[:, 0], Xt[:, 1], c='#e74c3c', s=8, alpha=0.5, edgecolors='none')
    axes[1].set_title(fr'Noisy data  $\mathbf{{x}}_{{t}}$,  '
                      fr'$t={t}$,  $\sqrt{{\bar{{\alpha}}_t}}={np.sqrt(alpha_bars[t]):.3f}$')
    for ax in axes:
        ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
        ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

interact(_plot_fwd_step,
         t=IntSlider(min=0, max=T-1, step=5, value=0, description='t:'));

<a id="part2"></a>
# Part 2: Score Functions

## 2.1 What is a score?

The **score function** of a distribution $p(\mathbf{x})$ is the gradient of its log-density:
$$\mathbf{s}(\mathbf{x}) := \nabla_{\mathbf{x}} \log p(\mathbf{x}).$$

The score *points in the direction of increasing probability density* — it is a vector field that, in a sense, shows where the data is. If you start from any point and follow the score (Langevin dynamics), you drift toward high-density regions.

**Example — Gaussian:** For $p(\mathbf{x}) = \mathcal{N}(\boldsymbol{\mu}, \sigma^2\mathbf{I})$:
$$\nabla_{\mathbf{x}} \log p(\mathbf{x}) = -\frac{\mathbf{x} - \boldsymbol{\mu}}{\sigma^2}.$$
The score points toward the mean, with magnitude inversely proportional to $\sigma^2$.

**Mixture of Gaussians:** For $p(\mathbf{x}) = \sum_k w_k \mathcal{N}(\boldsymbol{\mu}_k, \sigma^2\mathbf{I})$ the score is the *responsibility-weighted average* of per-component scores:
$$\nabla_{\mathbf{x}} \log p(\mathbf{x}) = \sum_{k} r_k(\mathbf{x})\cdot\left(-\frac{\mathbf{x}-\boldsymbol{\mu}_k}{\sigma^2}\right), \qquad r_k(\mathbf{x}) = \frac{w_k\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_k,\sigma^2\mathbf{I})}{\sum_j w_j\,\mathcal{N}(\mathbf{x};\boldsymbol{\mu}_j,\sigma^2\mathbf{I})}.$$

## 2.2 Score of the noisy marginal $q_t$

Because the forward process has a closed form, so does the noisy marginal. Starting from our 3-Gaussian data with per-cluster variance $\sigma_0^2$, the marginal at noise level $t$ is another mixture of Gaussians:

$$q_t(\mathbf{x}) = \frac{1}{3}\sum_{k=1}^{3} \mathcal{N}\!\left(\mathbf{x};\; \sqrt{\bar{\alpha}_t}\,\boldsymbol{\mu}_k,\; \sigma_t^2\,\mathbf{I}\right), \qquad \sigma_t^2 = \bar{\alpha}_t\sigma_0^2 + (1-\bar{\alpha}_t).$$

We can therefore compute the **true score** $\nabla_\mathbf{x} \log q_t(\mathbf{x})$ analytically — no network needed. This is what makes our toy dataset pedagogically ideal.

In [ ]:
def log_gauss(x, mu, var):
    # log N(x; mu, var*I),  x: (N,2), mu: (2,)
    diff = x - mu
    return -0.5 * np.sum(diff**2, axis=1) / var - np.log(2 * np.pi * var)

def score_qt(x, t):
    # Analytical score of q_t for our 3-Gaussian dataset
    ab   = alpha_bars[t]
    var  = ab * SIGMA0**2 + (1.0 - ab)          # σ_t²
    nmu  = np.sqrt(ab) * MEANS                   # scaled cluster centres (3,2)

    log_p = np.stack([log_gauss(x, nmu[k], var) for k in range(3)], axis=0)  # (3,N)
    log_r = log_p - logsumexp(log_p, axis=0, keepdims=True)                   # log-responsibilities
    r = np.exp(log_r)                                                          # (3,N)

    score = np.zeros_like(x)
    for k in range(3):
        score += r[k, :, np.newaxis] * (-(x - nmu[k]) / var)
    return score

# ── Visualise score fields at three noise levels ─────────────────────────────
grid_1d = np.linspace(-3.2, 3.2, 22)
gx, gy  = np.meshgrid(grid_1d, grid_1d)
grid    = np.c_[gx.ravel(), gy.ravel()]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
ts_score = [0, 80, 300]
titles   = [r'$t=0$  (clean data score)',
            r'$t=80$  (mild noise)',
            r'$t=300$  (heavy noise)']

for ax, t, title in zip(axes, ts_score, titles):
    Xt, _  = forward_sample(X0, t)
    s      = score_qt(grid, t)
    mag    = np.linalg.norm(s, axis=1).reshape(gx.shape)
    su, sv = s[:, 0].reshape(gx.shape), s[:, 1].reshape(gx.shape)
    # Normalise arrow length for readability; colour by magnitude
    norm   = np.sqrt(su**2 + sv**2 + 1e-8)
    ax.quiver(gx, gy, su/norm, sv/norm, mag, cmap='plasma',
              scale=30, alpha=0.75, width=0.004)
    ax.scatter(Xt[:300, 0], Xt[:300, 1], c='#3498db', s=4, alpha=0.35, edgecolors='none')
    ax.set_xlim(-3.2, 3.2); ax.set_ylim(-3.2, 3.2)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=11)

fig.suptitle(r'Score field $\nabla_\mathbf{x}\log q_t(\mathbf{x})$'
             '  (arrows = direction, colour = magnitude)', fontsize=12)
plt.tight_layout(); plt.show()

**Observations:**
- At $t=0$ the score field has three sharp "sinks" pointing toward the cluster centres; the field is strong and well-localised.
- As $t$ grows, noise blurs the clusters together. The score field becomes smoother and, for large $t$, converges to $-\mathbf{x}$ (the score of $\mathcal{N}(\mathbf{0},\mathbf{I})$).
- This smooth-to-sharp transition is the key intuition: **at large $t$ the score is easy to estimate** (it is nearly Gaussian); at small $t$ it requires distinguishing fine structure. A learned network $\mathbf{s}_\theta(\mathbf{x}_t, t)$ must handle the full range.

## 2.3 Langevin dynamics: sampling via the score

Given the score, we can sample from $p$ via **annealed Langevin dynamics**:
$$\mathbf{x}^{(k+1)} = \mathbf{x}^{(k)} + \frac{\eta}{2}\,\nabla_{\mathbf{x}}\log p(\mathbf{x}^{(k)}) + \sqrt{\eta}\,\boldsymbol{\xi}^{(k)}, \qquad \boldsymbol{\xi}^{(k)} \sim \mathcal{N}(\mathbf{0},\mathbf{I}).$$
As $\eta \to 0$ and steps $\to \infty$, the chain converges to $p$. The stochastic term prevents collapse to a mode.

**Experiment:** The code below runs Langevin dynamics with the *analytical* score $\nabla_\mathbf{x}\log q_0$ on our toy distribution.

In [ ]:
def langevin_sample(score_fn, n_samples=300, n_steps=800, eta=0.02, seed=1):
    rng_ = np.random.default_rng(seed)
    x = rng_.standard_normal((n_samples, 2)) * 2.0   # initialise from wide Gaussian
    snaps = {0: x.copy()}
    for k in range(1, n_steps + 1):
        s = score_fn(x)
        x = x + (eta / 2) * s + np.sqrt(eta) * rng_.standard_normal(x.shape)
        if k in (50, 200, 500, n_steps):
            snaps[k] = x.copy()
    return snaps

snaps = langevin_sample(lambda x: score_qt(x, 0))

fig, axes = plt.subplots(1, 4, figsize=(13, 3.3))
for ax, (step, Xs) in zip(axes, snaps.items()):
    ax.scatter(Xs[:, 0], Xs[:, 1], c='#9b59b6', s=7, alpha=0.55, edgecolors='none')
    # Overlay true cluster centres
    ax.scatter(MEANS[:, 0], MEANS[:, 1], c='gold', s=80, edgecolors='k', lw=0.8,
               zorder=5, marker='*')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'step {step}', fontsize=11)

fig.suptitle(r'Langevin dynamics with analytical score $\nabla_\mathbf{x}\log q_0$'
             '  (★ = cluster centres)', fontsize=12)
plt.tight_layout(); plt.show()

<a id="part3"></a>
# Part 3: The Reverse Process and DDPM

## 3.1 Reversing the forward chain

Starting from noise $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0},\mathbf{I})$, we want to run the Markov chain *backwards* to recover a sample from the data. The reverse transition is:
$$q(\mathbf{x}_{t-1} \mid \mathbf{x}_t, \mathbf{x}_0) = \mathcal{N}\!\left(\mathbf{x}_{t-1};\; \tilde{\boldsymbol{\mu}}_t(\mathbf{x}_t, \mathbf{x}_0),\; \tilde{\beta}_t\,\mathbf{I}\right),$$
$$\tilde{\boldsymbol{\mu}}_t = \frac{\sqrt{\bar{\alpha}_{t-1}}\,\beta_t}{1-\bar{\alpha}_t}\,\mathbf{x}_0 + \frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t}\,\mathbf{x}_t, \qquad \tilde{\beta}_t = \frac{1-\bar{\alpha}_{t-1}}{1-\bar{\alpha}_t}\,\beta_t.$$

The problem: this requires knowing $\mathbf{x}_0$, which is exactly what we are trying to generate!

## 3.2 DDPM: predict the noise, reconstruct $\mathbf{x}_0$

Ho et al. (2020) train a network $\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$ to **predict the noise** $\boldsymbol{\epsilon}$ that was added at step $t$. Given the prediction, we estimate:
$$\hat{\mathbf{x}}_0 = \frac{\mathbf{x}_t - \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t,t)}{\sqrt{\bar{\alpha}_t}},$$
and substitute into $\tilde{\boldsymbol{\mu}}_t$. This simplifies to:

$$\boxed{\boldsymbol{\mu}_\theta(\mathbf{x}_t, t) = \frac{1}{\sqrt{\alpha_t}}\!\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\,\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)\right).}$$

**Connection to score:** The predicted noise and the score are related by
$$\boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t) \approx -\sqrt{1-\bar{\alpha}_t}\;\nabla_{\mathbf{x}_t}\log q_t(\mathbf{x}_t).$$
Predicting noise is equivalent to estimating the score — DDPM and score-based models are two views of the same idea.

## 3.3 Sampling algorithm

Starting from $\mathbf{x}_T \sim \mathcal{N}(\mathbf{0},\mathbf{I})$, iterate for $t = T, T-1, \ldots, 1$:
$$\mathbf{x}_{t-1} = \boldsymbol{\mu}_\theta(\mathbf{x}_t, t) + \sqrt{\tilde{\beta}_t}\,\mathbf{z}, \qquad \mathbf{z} \sim \mathcal{N}(\mathbf{0},\mathbf{I}) \text{ if } t > 1 \text{, else } \mathbf{z}=\mathbf{0}.$$

In [ ]:
def ddpm_reverse(n_samples=300, seed=2):
    # Use the analytical score in place of a trained network:
    # eps_hat(x_t, t) = -sqrt(1 - abar_t) * score_qt(x_t, t)
    rng_ = np.random.default_rng(seed)
    x = rng_.standard_normal((n_samples, 2))   # x_T ~ N(0,I)
    snaps = {T: x.copy()}

    for t in range(T - 1, -1, -1):
        ab_t   = alpha_bars[t]
        s_t    = score_qt(x, t)
        eps_hat = -np.sqrt(1.0 - ab_t) * s_t    # analytical noise prediction

        mu = (x - betas[t] / np.sqrt(1.0 - ab_t) * eps_hat) / np.sqrt(alphas[t])

        if t > 0:
            ab_prev   = alpha_bars[t - 1]
            beta_tilde = betas[t] * (1.0 - ab_prev) / (1.0 - ab_t)
            x = mu + np.sqrt(beta_tilde) * rng_.standard_normal(x.shape)
        else:
            x = mu

        if t in (300, 200, 100, 40, 0):
            snaps[t] = x.copy()

    return snaps

rev_snaps = ddpm_reverse()
steps_show = [T, 300, 200, 100, 40, 0]
labels     = [f't={s}' for s in steps_show]
labels[-1] = 't=0  (generated)'

fig, axes = plt.subplots(1, 6, figsize=(16, 3.0))
for ax, t, lbl in zip(axes, steps_show, labels):
    Xs = rev_snaps[t]
    ax.scatter(Xs[:, 0], Xs[:, 1], c='#2ecc71', s=7, alpha=0.55, edgecolors='none')
    ax.scatter(MEANS[:, 0], MEANS[:, 1], c='gold', s=60, edgecolors='k', lw=0.7,
               zorder=5, marker='*')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(lbl, fontsize=10)

fig.suptitle('DDPM reverse diffusion  (★ = true cluster centres)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def _plot_reverse_step(t=T):
    if t not in rev_snaps:
        # find nearest available snapshot
        available = sorted(rev_snaps.keys())
        t = min(available, key=lambda s: abs(s - t))
    Xs = rev_snaps[t]
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ax.scatter(Xs[:, 0], Xs[:, 1], c='#2ecc71', s=10, alpha=0.6, edgecolors='none')
    ax.scatter(MEANS[:, 0], MEANS[:, 1], c='gold', s=100, edgecolors='k',
               lw=0.8, zorder=5, marker='*')
    ax.set_xlim(-3.5, 3.5); ax.set_ylim(-3.5, 3.5)
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f'Reverse diffusion at t = {t}', fontsize=13)
    plt.tight_layout(); plt.show()

available_t = sorted(rev_snaps.keys(), reverse=True)
interact(_plot_reverse_step,
         t=IntSlider(min=0, max=T, step=1, value=T, description='t:'));

<a id="part4"></a>
# Part 4: Training — Denoising Score Matching

## 4.1 The training objective

In practice $\boldsymbol{\epsilon}_\theta$ is a neural network (a U-Net for images). Training minimises:

$$\boxed{\mathcal{L}_{\mathrm{DDPM}} = \mathbb{E}_{t,\,\mathbf{x}_0 \sim q_0,\,\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0},\mathbf{I})}\!\left[\left\|\boldsymbol{\epsilon} - \boldsymbol{\epsilon}_\theta\!\left(\sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon},\; t\right)\right\|^2\right].}$$

**Procedure** (one training step):
1. Sample $\mathbf{x}_0 \sim q_0$ (a training example), $t \sim \mathrm{Uniform}\{1, \ldots, T\}$, $\boldsymbol{\epsilon} \sim \mathcal{N}(\mathbf{0},\mathbf{I})$.
2. Compute $\mathbf{x}_t = \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0 + \sqrt{1-\bar{\alpha}_t}\,\boldsymbol{\epsilon}$ (one forward step, closed form).
3. Predict $\hat{\boldsymbol{\epsilon}} = \boldsymbol{\epsilon}_\theta(\mathbf{x}_t, t)$.
4. Gradient step on $\|\boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}}\|^2$.

This is a **denoising** objective — remarkably similar to the denoising autoencoders studied long before diffusion models. The insight of score matching is that this objective, when summed over all noise levels, is equivalent to matching $\boldsymbol{\epsilon}_\theta$ to the true score field.

## 4.2 Why denoising = score matching

The conditional score of the forward kernel satisfies:
$$\nabla_{\mathbf{x}_t}\log q(\mathbf{x}_t \mid \mathbf{x}_0) = -\frac{\mathbf{x}_t - \sqrt{\bar{\alpha}_t}\,\mathbf{x}_0}{1-\bar{\alpha}_t} = -\frac{\boldsymbol{\epsilon}}{\sqrt{1-\bar{\alpha}_t}}.$$
Minimising the denoising loss is equivalent to minimising $\mathbb{E}\big[\big\|\mathbf{s}_\theta(\mathbf{x}_t,t) - \nabla_{\mathbf{x}_t}\log q_t(\mathbf{x}_t)\big\|^2\big]$ (the true score matching objective), a result due to Vincent (2011).

> **Exercise 4.1.** *(Deriving $q(\mathbf{x}_t|\mathbf{x}_0)$)*
>
> **(a)** Prove the closed form $q(\mathbf{x}_t|\mathbf{x}_0) = \mathcal{N}(\sqrt{\bar\alpha_t}\mathbf{x}_0, (1-\bar\alpha_t)\mathbf{I})$ by induction on $t$. Use the fact that the sum of two independent Gaussians is Gaussian, and that $\mathbf{x}_t = \sqrt{\alpha_t}\mathbf{x}_{t-1} + \sqrt{\beta_t}\boldsymbol{\epsilon}$.
>
> **(b)** Show that as $t \to \infty$ with a linear schedule $\beta_t \in (0,1)$, we have $\bar\alpha_t \to 0$ and hence $q(\mathbf{x}_t|\mathbf{x}_0) \to \mathcal{N}(\mathbf{0},\mathbf{I})$.
>
> **(c)** Verify the identity $\nabla_{\mathbf{x}_t}\log q(\mathbf{x}_t|\mathbf{x}_0) = -\boldsymbol{\epsilon}/\sqrt{1-\bar\alpha_t}$ and confirm that this gives the equivalence between denoising and score matching.

In [ ]:
# Demonstrate the denoising objective on one random training step
rng2 = np.random.default_rng(99)
x0_batch = sample_data(128, seed=3)

losses = []
ts_all = rng2.integers(0, T, size=128)
for i in range(128):
    t_i = int(ts_all[i])
    xt, eps = forward_sample(x0_batch[[i]], t_i)
    # "perfect" denoiser: eps_hat = -sqrt(1-ab)*score = eps  (on expectation)
    eps_hat = -np.sqrt(1 - alpha_bars[t_i]) * score_qt(xt, t_i)
    losses.append(float(np.mean((eps - eps_hat)**2)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(ts_all, bins=30, color='#3498db', edgecolor='k', lw=0.4)
axes[0].set_xlabel('Sampled time-step $t$')
axes[0].set_ylabel('Count')
axes[0].set_title('Uniform sampling of $t$ during training')

axes[1].scatter(ts_all, losses, s=10, alpha=0.5, color='#e74c3c', edgecolors='none')
axes[1].set_xlabel('Time-step $t$')
axes[1].set_ylabel(r'$\|\boldsymbol{\epsilon} - \hat{\boldsymbol{\epsilon}}\|^2$')
axes[1].set_title('Denoising loss per sample (analytical predictor)')

plt.tight_layout(); plt.show()
print("Mean denoising loss:", np.mean(losses).round(4))

<a id="part5"></a>
# Part 5: Extensions and the Broader Picture

## 5.1 DDIM — deterministic sampling

The standard DDPM sampler adds fresh noise at each reverse step, requiring $T \approx 1000$ steps for high quality. **DDIM** (Song et al., 2021) finds a *non-Markovian* process that shares the same marginals $q_t$ but admits a deterministic reverse ODE:
$$\mathbf{x}_{t-1} = \sqrt{\bar\alpha_{t-1}}\underbrace{\frac{\mathbf{x}_t - \sqrt{1-\bar\alpha_t}\,\boldsymbol{\epsilon}_\theta}{\sqrt{\bar\alpha_t}}}_{\text{predicted }\mathbf{x}_0} + \sqrt{1-\bar\alpha_{t-1}}\,\boldsymbol{\epsilon}_\theta.$$
This allows generation in as few as 20–50 steps, accelerating diffusion by 50×.

## 5.2 Continuous time: score SDEs

Song et al. (2021) unify all of these variants. The forward process is modelled as an SDE:
$$d\mathbf{x} = \mathbf{f}(\mathbf{x},t)\,dt + g(t)\,d\mathbf{w},$$
and the reverse-time SDE (Anderson, 1982) is:
$$d\mathbf{x} = \bigl[\mathbf{f}(\mathbf{x},t) - g(t)^2\,\nabla_{\mathbf{x}}\log p_t(\mathbf{x})\bigr]\,dt + g(t)\,d\bar{\mathbf{w}}.$$
The score $\nabla_\mathbf{x}\log p_t(\mathbf{x})$ is exactly what the network $\mathbf{s}_\theta$ learns. Setting the noise term in the reverse SDE to zero gives an **ODE** (the probability-flow ODE), enabling likelihood evaluation and fast sampling.

| Paradigm | Training objective | Sampling | Key paper |
|----------|-------------------|----------|-----------|
| **DDPM** | Denoising $\|\boldsymbol\epsilon - \boldsymbol\epsilon_\theta\|^2$ | Stochastic, $T\sim1000$ steps | Ho et al. 2020 |
| **Score matching** | $\|\mathbf{s}_\theta - \nabla\log p_t\|^2$ | Annealed Langevin | Song & Ermon 2019 |
| **DDIM** | Same as DDPM | Deterministic ODE, 20–50 steps | Song et al. 2021 |
| **Score SDE** | Unified via SDE framework | SDE or ODE | Song et al. 2021 |

> **Exercise 5.1.** *(Score of a Gaussian)*
>
> **(a)** Verify that for $p(\mathbf{x}) = \mathcal{N}(\boldsymbol\mu, \sigma^2\mathbf{I})$, the score is $\nabla_\mathbf{x}\log p(\mathbf{x}) = -(\mathbf{x}-\boldsymbol\mu)/\sigma^2$.
>
> **(b)** What is $\lim_{t\to T}\nabla_\mathbf{x}\log q_t(\mathbf{x})$? Confirm your answer using `score_qt(x, T-1)` on a grid of points.
>
> **(c)** *(Challenge)* Implement a simplified DDIM sampler on our 2-D toy distribution, using fewer steps (say 40 instead of 400). Compare the quality of generated samples to DDPM.

---
# Summary

| Concept | Key formula / takeaway |
|---------|------------------------|
| **Forward process** | $q(\mathbf{x}_t\|\mathbf{x}_0) = \mathcal{N}(\sqrt{\bar\alpha_t}\mathbf{x}_0,\,(1{-}\bar\alpha_t)\mathbf{I})$ |
| **Reparameterisation** | $\mathbf{x}_t = \sqrt{\bar\alpha_t}\mathbf{x}_0 + \sqrt{1{-}\bar\alpha_t}\boldsymbol\epsilon$ |
| **Score function** | $\mathbf{s}(\mathbf{x}) = \nabla_\mathbf{x}\log p(\mathbf{x})$; points toward high-density regions |
| **DDPM objective** | $\mathbb{E}\|\boldsymbol\epsilon - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\|^2$ — denoising equals score matching |
| **Reverse step** | $\boldsymbol\mu_\theta = \frac{1}{\sqrt{\alpha_t}}(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1{-}\bar\alpha_t}}\boldsymbol\epsilon_\theta)$ |
| **Continuous limit** | Forward SDE + reverse SDE (via score) = score-based generative model |